In [1]:
# FFLUNet Training

In [2]:
import kagglehub
path = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
print(path)

Using Colab cache for faster access to the 'brats20-dataset-training-validation' dataset.
/kaggle/input/brats20-dataset-training-validation


In [3]:
import os
import glob

brats_path = os.path.join(
    path,
    "BraTS2020_TrainingData",
    "MICCAI_BraTS2020_TrainingData"
)

patients = sorted(
    glob.glob(
        os.path.join(
            brats_path,
            "BraTS20_*"
        )
    )
)

print("Patients:", len(patients))

Patients: 369


In [4]:
# to rename the 355 folder
# print(patients[354])
# print(os.listdir(patients[354]))
# print(os.listdir(patients[355]))
# print(patients[355])

# patient_change = patients[354]

# old_path = os.path.join(patient_change, "W39_1998.09.19_Segm.nii")
# new_path = os.path.join(patient_change, "BraTS20_Training_355_seg.nii")

# os.rename(old_path, new_path)
# print(os.listdir(patients[354]))

# read only so cannot edit
# drop instead

import os

patients = [
    p for p in patients
    if os.path.exists(
        os.path.join(p, f"{os.path.basename(p)}_seg.nii")
    )
]
print("Total patients:", len(patients))

Total patients: 368


In [5]:
import os

print(brats_path)

for item in sorted(os.listdir(brats_path))[:5]:
    print(item)

/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData
BraTS20_Training_001
BraTS20_Training_002
BraTS20_Training_003
BraTS20_Training_004
BraTS20_Training_005


In [6]:
import os

sample_patient = patients[0]

print(sample_patient)

for f in sorted(os.listdir(sample_patient)):
    print(f)

/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/BraTS20_Training_001
BraTS20_Training_001_flair.nii
BraTS20_Training_001_seg.nii
BraTS20_Training_001_t1.nii
BraTS20_Training_001_t1ce.nii
BraTS20_Training_001_t2.nii


In [7]:
!pip install -q nnunetv2 SimpleITK nibabel

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 9.6 MB/s eta 0:00:0

In [8]:
!nnUNetv2_plan_and_preprocess -h
import nnunetv2
print("nnU-Net imported successfully")

usage: nnUNetv2_plan_and_preprocess [-h] [-d D [D ...]] [-fpe FPE]
                                    [-npfp NPFP] [--verify_dataset_integrity]
                                    [--no_pp] [--clean] [-pl PL]
                                    [-gpu_memory_target GPU_MEMORY_TARGET]
                                    [-preprocessor_name PREPROCESSOR_NAME]
                                    [-overwrite_target_spacing OVERWRITE_TARGET_SPACING [OVERWRITE_TARGET_SPACING ...]]
                                    [-overwrite_plans_name OVERWRITE_PLANS_NAME]
                                    [-c C [C ...]] [-np NP [NP ...]]
                                    [--verbose] [--no_pbar]

options:
  -h, --help            show this help message and exit
  -d D [D ...]          [REQUIRED] List of dataset IDs. Example: 2 4 5. This
                        will run fingerprint extraction, experiment planning
                        and preprocessing for these datasets. Can of course
              

In [9]:
import os

nnunet_root = "/content/nnunet_workspace"

os.makedirs(nnunet_root, exist_ok=True)

print(nnunet_root)

/content/nnunet_workspace


In [10]:
dataset_dir = os.path.join(
    nnunet_root,
    "nnUNet_raw",
    "Dataset001_BraTS2020"
)

imagesTr = os.path.join(dataset_dir, "imagesTr")
labelsTr = os.path.join(dataset_dir, "labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

print(imagesTr)
print(labelsTr)

/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/imagesTr
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr


In [11]:
sample = patients[0]

print(os.path.basename(sample))

for f in sorted(os.listdir(sample)):
    print(f)

BraTS20_Training_001
BraTS20_Training_001_flair.nii
BraTS20_Training_001_seg.nii
BraTS20_Training_001_t1.nii
BraTS20_Training_001_t1ce.nii
BraTS20_Training_001_t2.nii


In [12]:
import shutil
from tqdm import tqdm

num_patients = 0

for patient_dir in tqdm(patients):
    patient_id = os.path.basename(patient_dir)

    flair = os.path.join(patient_dir, f"{patient_id}_flair.nii")
    t1    = os.path.join(patient_dir, f"{patient_id}_t1.nii")
    t1ce  = os.path.join(patient_dir, f"{patient_id}_t1ce.nii")
    t2    = os.path.join(patient_dir, f"{patient_id}_t2.nii")
    seg   = os.path.join(patient_dir, f"{patient_id}_seg.nii")

    # modalities
    shutil.copy2(
        flair,
        os.path.join(imagesTr, f"{patient_id}_0000.nii")
    )

    shutil.copy2(
        t1,
        os.path.join(imagesTr, f"{patient_id}_0001.nii")
    )

    shutil.copy2(
        t1ce,
        os.path.join(imagesTr, f"{patient_id}_0002.nii")
    )

    shutil.copy2(
        t2,
        os.path.join(imagesTr, f"{patient_id}_0003.nii")
    )

    # label
    shutil.copy2(
        seg,
        os.path.join(labelsTr, f"{patient_id}.nii")
    )

    num_patients += 1

print("Converted patients:", num_patients)

100%|██████████| 368/368 [06:12<00:00,  1.01s/it]

Converted patients: 368


In [13]:
print("imagesTr:", len(os.listdir(imagesTr)))
print("labelsTr:", len(os.listdir(labelsTr)))

imagesTr: 1472
labelsTr: 368


In [14]:
import nibabel as nib
import numpy as np

sample_mask = os.path.join(
    labelsTr,
    os.listdir(labelsTr)[0]
)

mask = nib.load(sample_mask).get_fdata()

print(np.unique(mask))

[0. 1. 2. 4.]


In [15]:
import os
import json

dataset_json = {
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1CE",
        "3": "T2"
    },

    "labels": {
        "background": 0,
        "NCR_NET": 1,
        "ED": 2,
        "ET": 3
    },

    "numTraining": len(os.listdir(labelsTr)),

    "file_ending": ".nii"
}

json_path = os.path.join(dataset_dir, "dataset.json")

with open(json_path, "w") as f:
    json.dump(dataset_json, f, indent=4)

print(json_path)

/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/dataset.json


In [16]:
with open(json_path) as f:
    print(f.read())

{
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1CE",
        "3": "T2"
    },
    "labels": {
        "background": 0,
        "NCR_NET": 1,
        "ED": 2,
        "ET": 3
    },
    "numTraining": 368,
    "file_ending": ".nii"
}


In [17]:
import os

nnUNet_raw = os.path.join(nnunet_root, "nnUNet_raw")
nnUNet_preprocessed = os.path.join(nnunet_root, "nnUNet_preprocessed")
nnUNet_results = os.path.join(nnunet_root, "nnUNet_results")

os.makedirs(nnUNet_preprocessed, exist_ok=True)
os.makedirs(nnUNet_results, exist_ok=True)

print(nnUNet_raw)
print(nnUNet_preprocessed)
print(nnUNet_results)

/content/nnunet_workspace/nnUNet_raw
/content/nnunet_workspace/nnUNet_preprocessed
/content/nnunet_workspace/nnUNet_results


In [18]:
import os

os.environ["nnUNet_raw"] = nnUNet_raw
os.environ["nnUNet_preprocessed"] = nnUNet_preprocessed
os.environ["nnUNet_results"] = nnUNet_results

print(os.environ["nnUNet_raw"])
print(os.environ["nnUNet_preprocessed"])
print(os.environ["nnUNet_results"])

/content/nnunet_workspace/nnUNet_raw
/content/nnunet_workspace/nnUNet_preprocessed
/content/nnunet_workspace/nnUNet_results


In [19]:
!ls /content/nnunet_workspace/nnUNet_raw

Dataset001_BraTS2020


In [20]:
import os
import nibabel as nib
import numpy as np

sample_file = os.path.join(labelsTr, os.listdir(labelsTr)[0])

nii = nib.load(sample_file)

mask = nii.get_fdata()

print("Before:", np.unique(mask))

mask[mask == 4] = 3

print("After :", np.unique(mask))

Before: [0. 1. 2. 4.]
After : [0. 1. 2. 3.]


In [21]:
import os
import nibabel as nib
import numpy as np
from tqdm import tqdm

label_files = sorted(os.listdir(labelsTr))

for fname in tqdm(label_files):
    fpath = os.path.join(labelsTr, fname)

    nii = nib.load(fpath)

    mask = nii.get_fdata()

    mask[mask == 4] = 3

    mask = mask.astype(np.uint8)

    new_nii = nib.Nifti1Image(
        mask,
        affine=nii.affine,
        header=nii.header
    )

    nib.save(new_nii, fpath)

print("Done")

100%|██████████| 368/368 [00:46<00:00,  7.89it/s]

Done


In [22]:
print(mask.dtype)

uint8


In [23]:
import random
import nibabel as nib
import numpy as np

for fname in random.sample(os.listdir(labelsTr), 3):
    mask = nib.load(os.path.join(labelsTr, fname)).get_fdata()
    print(fname, np.unique(mask))

BraTS20_Training_105.nii [0. 1. 2. 3.]
BraTS20_Training_247.nii [0. 1. 2. 3.]
BraTS20_Training_065.nii [0. 1. 2. 3.]


In [24]:
with open(json_path) as f:
    print(f.read())

{
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1CE",
        "3": "T2"
    },
    "labels": {
        "background": 0,
        "NCR_NET": 1,
        "ED": 2,
        "ET": 3
    },
    "numTraining": 368,
    "file_ending": ".nii"
}


In [20]:
!nnUNetv2_extract_fingerprint -d 1

Dataset001_BraTS2020
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_extract_fingerprint", line 8, in <module>
    sys.exit(extract_fingerprint_entry())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_entrypoints.py", line 60, in extract_fingerprint_entry
    extract_fingerprints(
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 68, in extract_fingerprints
    extract_fingerprint_dataset(
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 47, in extract_fingerprint_dataset
    return fpe.run(overwrite_existing=clean)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FFLUNet/nnunetv2/experiment_planning/dataset_fingerprint/fingerprint_extractor.py", line 157, in run
    reader_writer_class = determine_reader_writer_from_dataset_json(
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/conte

In [23]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS2020
Using <class 'nnunetv2.imageio.nibabel_reader_writer.NibabelIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.nibabel_reader_writer.NibabelIO'> reader/writer
100% 368/368 [07:43<00:00,  1.26s/it]
Experiment planning...
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 105, 'patch_size': array([192, 160]), 'median_image_size_in_voxels': array([170., 138.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'UNet_class_name': 'PlainConvUNet', 'UNet_base_num_features': 32, 'n_conv_per_stage_encoder': (2, 2, 2, 2, 2, 2), 'n_conv_per_stage_decoder': (2, 2, 2, 2, 2), 'num_pool_per_axis': 

In [1]:
!du -sh /content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/*

56K	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/dataset_fingerprint.json
4.0K	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/dataset.json
4.1G	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/gt_segmentations
5.6G	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d
5.3G	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_3d_fullres
12K	/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans.json


In [2]:
!df -h

Filesystem                                                                                                                            Size  Used Avail Use% Mounted on
overlay                                                                                                                               113G   94G   20G  83% /
tmpfs                                                                                                                                  64M     0   64M   0% /dev
shm                                                                                                                                   5.7G     0  5.7G   0% /dev/shm
/dev/root                                                                                                                             2.0G  1.3G  696M  65% /usr/sbin/docker-init
/dev/sda1                                                                                                                             119G   98G   22G  83% /opt/bin/.nvidi

In [29]:
!git clone https://github.com/Dutta-SD/FFLUNet.git
%cd FFLUNet
!pip install -e .

Cloning into 'FFLUNet'...
remote: Enumerating objects: 686, done.
remote: Counting objects: 100% (686/686), done.
remote: Compressing objects: 100% (459/459), done.
remote: Total 686 (delta 381), reused 506 (delta 225), pack-reused 0 (from 0)
Receiving objects: 100% (686/686), 14.21 MiB | 11.60 MiB/s, done.
Resolving deltas: 100% (381/381), done.
/content/FFLUNet
Obtaining file:///content/FFLUNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tifffile to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of imagecodecs to determine which version is compatible with other requirements. This could take a while.
  Using cached argp

In [3]:
import nnunetv2
print(nnunetv2.__file__)

/content/FFLUNet/nnunetv2/__init__.py


In [5]:
!ls -la /content

total 24
drwxr-xr-x 1 root root 4096 Jun 26 17:51 .
drwxr-xr-x 1 root root 4096 Jun 26 16:40 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:39 .config
drwxr-xr-x 7 root root 4096 Jun 26 17:51 FFLUNet
drwxr-xr-x 5 root root 4096 Jun 26 17:07 nnunet_workspace
drwxr-xr-x 1 root root 4096 Jun  4 13:39 sample_data


In [6]:
import os

os.environ["nnUNet_raw"] = "/content/nnunet_workspace/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnunet_workspace/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnunet_workspace/nnUNet_results"

print(os.environ["nnUNet_raw"])
print(os.environ["nnUNet_preprocessed"])
print(os.environ["nnUNet_results"])

/content/nnunet_workspace/nnUNet_raw
/content/nnunet_workspace/nnUNet_preprocessed
/content/nnunet_workspace/nnUNet_results


In [8]:
import torch
import nnunetv2

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("nnUNet:", nnunetv2.__version__)

Torch: 2.11.0+cu128
CUDA: 12.8


AttributeError: module 'nnunetv2' has no attribute '__version__'

In [9]:
!sed -n '1,200p' /content/FFLUNet/nnunetv2/training/lr_scheduler/polylr.py

from torch.optim.lr_scheduler import _LRScheduler


class PolyLRScheduler(_LRScheduler):
    def __init__(
        self,
        optimizer,
        initial_lr: float,
        max_steps: int,
        exponent: float = 0.9,
        current_step: int = None,
    ):
        self.optimizer = optimizer
        self.initial_lr = initial_lr
        self.max_steps = max_steps
        self.exponent = exponent
        self.ctr = 0
        super().__init__(
            optimizer, current_step if current_step is not None else -1, False
        )

    def step(self, current_step=None):
        if current_step is None or current_step == -1:
            current_step = self.ctr
            self.ctr += 1

        new_lr = self.initial_lr * (1 - current_step / self.max_steps) ** self.exponent
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = new_lr


In [10]:
from pathlib import Path

path = Path("/content/FFLUNet/nnunetv2/training/lr_scheduler/polylr.py")

text = path.read_text()

old = """super().__init__(
            optimizer, current_step if current_step is not None else -1, False
        )"""

new = """super().__init__(
            optimizer,
            last_epoch=current_step if current_step is not None else -1,
        )"""

text = text.replace(old, new)

path.write_text(text)

print("Patched successfully!")

Patched successfully!


In [11]:
!sed -n '1,40p' /content/FFLUNet/nnunetv2/training/lr_scheduler/polylr.py

from torch.optim.lr_scheduler import _LRScheduler


class PolyLRScheduler(_LRScheduler):
    def __init__(
        self,
        optimizer,
        initial_lr: float,
        max_steps: int,
        exponent: float = 0.9,
        current_step: int = None,
    ):
        self.optimizer = optimizer
        self.initial_lr = initial_lr
        self.max_steps = max_steps
        self.exponent = exponent
        self.ctr = 0
        super().__init__(
            optimizer,
            last_epoch=current_step if current_step is not None else -1,
        )

    def step(self, current_step=None):
        if current_step is None or current_step == -1:
            current_step = self.ctr
            self.ctr += 1

        new_lr = self.initial_lr * (1 - current_step / self.max_steps) ** self.exponent
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = new_lr


In [13]:
!find /content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020 -maxdepth 2

/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/dataset.json
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_199.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_364.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_145.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_130.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_180.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_280.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_342.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/BraTS20_Training_311.nii
/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/labelsTr/B

In [14]:
!find /content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020 -maxdepth 2

/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_278.b2nd
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_173.b2nd
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_092.pkl
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_204.pkl
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_101.pkl
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_270_seg.b2nd
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_Training_352_seg.b2nd
/content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_2d/BraTS20_

In [15]:
!ls /content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_3d_fullres | wc -l

1104


In [16]:
!ls /content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020/nnUNetPlans_3d_fullres | head

BraTS20_Training_001.b2nd
BraTS20_Training_001.pkl
BraTS20_Training_001_seg.b2nd
BraTS20_Training_002.b2nd
BraTS20_Training_002.pkl
BraTS20_Training_002_seg.b2nd
BraTS20_Training_003.b2nd
BraTS20_Training_003.pkl
BraTS20_Training_003_seg.b2nd
BraTS20_Training_004.b2nd


In [ ]:
!nnUNetv2_train 1 3d_fullres 0 -tr nnUNetTrainer_FFLUNet

2026-06-26 18:53:50.606527: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using device: cuda:0
/content/FFLUNet/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:233: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == "cuda" else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#########################################################

In [21]:
!ls /content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/imagesTr | head

BraTS20_Training_001_0000.nii
BraTS20_Training_001_0001.nii
BraTS20_Training_001_0002.nii
BraTS20_Training_001_0003.nii
BraTS20_Training_002_0000.nii
BraTS20_Training_002_0001.nii
BraTS20_Training_002_0002.nii
BraTS20_Training_002_0003.nii
BraTS20_Training_003_0000.nii
BraTS20_Training_003_0001.nii


In [22]:
import json

path = "/content/nnunet_workspace/nnUNet_raw/Dataset001_BraTS2020/dataset.json"

with open(path) as f:
    d = json.load(f)

d["overwrite_image_reader_writer"] = "NibabelIO"

with open(path, "w") as f:
    json.dump(d, f, indent=4)

print("Updated dataset.json")

Updated dataset.json


In [ ]:
!nnUNetv2_train 1 3d_fullres 0


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-16 04:08:03.536474: Using torch.compile...
2026-06-16 04:08:06.964715: do_dummy_2d_data_aug: False
2026-06-16 04:08:06.966641: Creating new 5-fold cross-validation split...
2026-06-16 04:08:06.972063: Desired fold for training: 0
2026-06-16 04:08:06.972260: This split has 294 training an

In [ ]:
import inspect
import nnunetv2.training.nnUNetTrainer.nnUNetTrainer as trainer_module

print(trainer_module.__file__)

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
!pip show nnunetv2

Name: nnunetv2
Version: 2.8.0
Summary: nnU-Net is a framework for out-of-the box image segmentation.
Home-page: https://github.com/MIC-DKFZ/nnUNet
Author: Helmholtz Imaging Applied Computer Vision Lab
Author-email: Fabian Isensee <f.isensee@dkfz-heidelberg.de>
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i)

In [ ]:
!find /usr/local/lib/python3.12/dist-packages/nnunetv2 -name "nnUNetTrainer.py"

/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py


In [ ]:
!grep -n "num_epochs" /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py

159:        self.num_epochs = 1000
271:                "num_epochs": self.num_epochs,
555:        lr_scheduler = PolyLRScheduler(optimizer, self.initial_lr, self.num_epochs)
1181:        if (current_epoch + 1) % self.save_every == 0 and current_epoch != (self.num_epochs - 1):
1420:        for epoch in range(self.current_epoch, self.num_epochs):


In [ ]:
!find /usr/local/lib/python3.12/dist-packages/nnunetv2 -type f | grep -i supervision

/usr/local/lib/python3.12/dist-packages/nnunetv2/training/loss/__pycache__/deep_supervision.cpython-312.pyc
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/loss/deep_supervision.py
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/data_augmentation/custom_transforms/__pycache__/deep_supervision_donwsampling.cpython-312.pyc
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/data_augmentation/custom_transforms/deep_supervision_donwsampling.py
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/variants/network_architecture/nnUNetTrainerNoDeepSupervision.py
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/variants/network_architecture/__pycache__/nnUNetTrainerNoDeepSupervision.cpython-312.pyc


In [ ]:
!grep -R "DeepSupervision" /usr/local/lib/python3.12/dist-packages/nnunetv2 | head -20

grep: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/loss/__pycache__/deep_supervision.cpython-312.pyc: binary file matches
grep: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/__pycache__/nnUNetTrainer.cpython-312.pyc: binary file matches
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/loss/deep_supervision.py:class DeepSupervisionWrapper(nn.Module):
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/loss/deep_supervision.py:        super(DeepSupervisionWrapper, self).__init__()
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:            loss = DeepSupervisionWrapper(loss, weights)
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/variants/loss/nnUNetTrainerTopkLoss.py:from nnunetv2.training.loss.deep_supervis

In [ ]:
!grep -R "deep_supervision" /usr/local/lib/python3.12/dist-packages/nnunetv2 | head -50

grep: /usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/experiment_planners/residual_unets/__pycache__/residual_encoder_unet_planners.cpython-312.pyc: binary file matches
grep: /usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/experiment_planners/__pycache__/resencUNet_planner.cpython-312.pyc: binary file matches
/usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/experiment_planners/residual_unets/residual_encoder_unet_planners.py:                              nonlin=nn.LeakyReLU, nonlin_kwargs={'inplace': True}, deep_supervision=True)
/usr/local/lib/python3.12/dist-packages/nnunetv2/experiment_planning/experiment_planners/residual_unets/residual_encoder_unet_planners.py:                              nonlin=nn.LeakyReLU, nonlin_kwargs={'inplace': True}, deep_supervision=True)
grep: /usr/local/lib/python3.12/dist-packages/nnunetv2/inference/__pycache__/predict_from_raw_data.cpython-312.pyc: binary file matches
/usr/local/lib/python

In [17]:
!which nnUNetv2_plan_and_preprocess
!python -c "import nnunetv2; print(nnunetv2.__file__)"

/usr/local/bin/nnUNetv2_plan_and_preprocess
/content/FFLUNet/nnunetv2/__init__.py


In [18]:
!rm -rf /content/nnunet_workspace/nnUNet_preprocessed/Dataset001_BraTS2020

In [19]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS2020
Traceback (most recent call last):
  File "/usr/local/bin/nnUNetv2_plan_and_preprocess", line 8, in <module>
    sys.exit(plan_and_preprocess_entry())
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_entrypoints.py", line 352, in plan_and_preprocess_entry
    extract_fingerprints(
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 68, in extract_fingerprints
    extract_fingerprint_dataset(
  File "/content/FFLUNet/nnunetv2/experiment_planning/plan_and_preprocess_api.py", line 44, in extract_fingerprint_dataset
    verify_dataset_integrity(join(nnUNet_raw, dataset_name), num_processes)
  File "/content/FFLUNet/nnunetv2/experiment_planning/verify_dataset_integrity.py", line 247, in verify_dataset_integrity
    reader_writer_class = determine_reader_writer_from_dataset_json(
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
import os

nnunet_raw = "/kaggle/working/nnUNet_raw"
dataset_name = "Dataset001_BraTS"

base = os.path.join(
    nnunet_raw,
    dataset_name
)

imagesTr = os.path.join(base,"imagesTr")
labelsTr = os.path.join(base,"labelsTr")

os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

print("Folders created")

Folders created


In [ ]:
import os
import shutil
import nibabel as nib
import numpy as np
from tqdm import tqdm

# Remove previous broken dataset
shutil.rmtree(
    "/kaggle/working/nnUNet_raw/Dataset001_BraTS",
    ignore_errors=True
)

# Recreate folders
base="/kaggle/working/nnUNet_raw/Dataset001_BraTS"

imagesTr=os.path.join(base,"imagesTr")
labelsTr=os.path.join(base,"labelsTr")

os.makedirs(imagesTr,exist_ok=True)
os.makedirs(labelsTr,exist_ok=True)

valid_cases=0
skipped=[]

for p in tqdm(patients):

    patient_id=os.path.basename(p)

    flair=os.path.join(
        p,
        f"{patient_id}_flair.nii"
    )

    t1=os.path.join(
        p,
        f"{patient_id}_t1.nii"
    )

    t1ce=os.path.join(
        p,
        f"{patient_id}_t1ce.nii"
    )

    t2=os.path.join(
        p,
        f"{patient_id}_t2.nii"
    )

    seg=os.path.join(
        p,
        f"{patient_id}_seg.nii"
    )

    files=[flair,t1,t1ce,t2,seg]

    if not all(os.path.exists(f) for f in files):
        skipped.append(patient_id)
        continue

    case_id=f"BraTS_{valid_cases:03d}"

    # Copy MRI modalities
    shutil.copy(
        flair,
        os.path.join(imagesTr,f"{case_id}_0000.nii.gz")
    )

    shutil.copy(
        t1,
        os.path.join(imagesTr,f"{case_id}_0001.nii.gz")
    )

    shutil.copy(
        t1ce,
        os.path.join(imagesTr,f"{case_id}_0002.nii.gz")
    )

    shutil.copy(
        t2,
        os.path.join(imagesTr,f"{case_id}_0003.nii.gz")
    )

    # Load segmentation
    img=nib.load(seg)
    mask=img.get_fdata()

    # Remap 4 -> 3
    mask=np.where(mask==4,3,mask)
    mask=mask.astype(np.uint8)

    corrected=nib.Nifti1Image(
        mask,
        img.affine,
        img.header
    )

    nib.save(
        corrected,
        os.path.join(
            labelsTr,
            f"{case_id}.nii.gz"
        )
    )

    valid_cases+=1

print("Valid:",valid_cases)
print("Skipped:",skipped)

100%|██████████| 369/369 [06:51<00:00,  1.11s/it]

Valid: 368
Skipped: ['BraTS20_Training_355']


In [ ]:
import json

dataset_json={

"name":"BraTS",

"channel_names":{
"0":"FLAIR",
"1":"T1",
"2":"T1ce",
"3":"T2"
},

"labels":{
"background":0,
"necrotic":1,
"edema":2,
"enhancing":3
},

"numTraining":valid_cases,

"file_ending":".nii.gz"
}

with open(
    os.path.join(base,"dataset.json"),
    "w"
) as f:

    json.dump(
        dataset_json,
        f,
        indent=4
    )

print("dataset.json updated")

dataset.json updated


In [ ]:
import os

os.environ["nnUNet_raw"]="/kaggle/working/nnUNet_raw"
os.environ["nnUNet_preprocessed"]="/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"]="/kaggle/working/nnUNet_results"

In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 368/368 [06:41<00:00,  1.09s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 170. 138.], 3d_lowres: [139, 170, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_si

In [ ]:
!nnUNetv2_train 1 3d_fullres 0


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/nnUNetTrainer.py:170: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  self.grad_scaler = (GradScaler("cuda") if not TORCH_HAS_OLD_GRADSCALER else GradScaler()) if self.device.type == 'cuda' else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#####################################################